In [ ]:
# remove top two lines with header information
import pathlib

data_dir = 'benchmarks'
upload_dir = 'data'

path = pathlib.Path(data_dir)
for file in path.iterdir():
    lines = None
    with open(f'{data_dir}/{file.name}', 'r') as f:
        lines = f.readlines()
        lines.pop(0)
        lines.pop(0)

    with open(f'{data_dir}/{file.name}', 'w') as f:
         f.writelines(lines)

In [ ]:
# easy copy and paste names
import pathlib

data_dir = 'benchmarks'
path = pathlib.Path(data_dir)

for file in path.iterdir():
    print(f'{data_dir}/{file.name}')

In [ ]:
# create plot by copying in files from above
# fps,frametime,cpu_load,cpu_power,gpu_load,cpu_temp,gpu_temp,gpu_core_clock,gpu_mem_clock,gpu_vram_used,gpu_power,ram_used,swap_used,process_rss,cpu_mhz,elapsed
import pandas as pd
from matplotlib import pyplot as plt

SECS_IN_MIN = 60
COLS = 2
ROWS = 4

file = 'benchmarks/mgs3_2026-03-14_16-20-41.csv'
df = pd.read_csv(file)
game = file.split('/')[1].split('_')[0]

params = ['fps', 'frametime', 'cpu_load', 'gpu_load', 'cpu_temp', 'ram_used', 'process_rss']


downsampled_dfs = {}
for param in params:
    df.index = pd.to_timedelta(df['elapsed'], unit='ns')
    df[f'{param}_smooth'] = df[param].rolling(window=3, center=True).mean()

    downsampled_df = df.resample('10s').mean()

    df['time_minutes'] = pd.to_timedelta(df['elapsed'], unit='ns').dt.total_seconds() / SECS_IN_MIN
    downsampled_df['time_minutes'] = pd.to_timedelta(downsampled_df['elapsed'], unit='ns').dt.total_seconds() / SECS_IN_MIN

    downsampled_dfs[param] = downsampled_df

fig, axs = plt.subplots(ncols=COLS, nrows=ROWS, figsize=(12, 10), layout="constrained")
fig.suptitle(f"{game} on Youyeetoo X1 Performance", fontweight='bold', fontsize=14)

fig.delaxes(axs[3][1])

for r in range(ROWS):
    for c in range(COLS):
        axs[r][c].set_xlabel('Time (Minutes)')

# FPS
axs[0][0].set_title('FPS')
axs[0][0].set_ylabel('FPS')
axs[0][0].set_ylim(0, 190)
axs[0][0].plot(df['time_minutes'], df['fps'], alpha=0.3, color="#B2AFAF", linewidth=0.4)
axs[0][0].plot(df['time_minutes'], df['fps_smooth'], alpha=0.3, color="#6E6E6E", linewidth=0.4)
axs[0][0].plot(downsampled_dfs['fps']['time_minutes'], downsampled_dfs['fps']['fps_smooth'], color="#2563EB", linewidth=2.5)

# Frametime
axs[0][1].set_title('Frametime')
axs[0][1].set_ylabel('Frametime (ms)')
axs[0][1].set_ylim(0, 40)
axs[0][1].plot(df['time_minutes'], df['frametime'], alpha=0.3, color="#B2AFAF", linewidth=0.4)
axs[0][1].plot(df['time_minutes'], df['frametime_smooth'], alpha=0.3, color="#6E6E6E", linewidth=0.4)
axs[0][1].plot(downsampled_dfs['frametime']['time_minutes'], downsampled_dfs['frametime']['frametime_smooth'], color="#2563EB", linewidth=2.5)

# RAM
axs[1][0].set_title('RAM Used')
axs[1][0].set_ylabel('RAM Used (GiB)')
axs[1][0].plot(df['time_minutes'], df['ram_used'], alpha=0.3, color="#B2AFAF", linewidth=0.4)
axs[1][0].plot(df['time_minutes'], df['ram_used_smooth'], alpha=0.3, color="#6E6E6E", linewidth=0.4)
axs[1][0].plot(downsampled_dfs['ram_used']['time_minutes'], downsampled_dfs['ram_used']['ram_used_smooth'], color="#2563EB", linewidth=2.5)

# Proc RAM
axs[1][1].set_title('Process RAM Used')
axs[1][1].set_ylabel('Process RAM Used (GiB)')
axs[1][1].plot(df['time_minutes'], df['process_rss'], alpha=0.3, color="#B2AFAF", linewidth=0.4)
axs[1][1].plot(df['time_minutes'], df['process_rss_smooth'], alpha=0.3, color="#6E6E6E", linewidth=0.4)
axs[1][1].plot(downsampled_dfs['process_rss']['time_minutes'], downsampled_dfs['process_rss']['process_rss_smooth'], color="#2563EB", linewidth=2.5)

# CPU Load
axs[2][0].set_title('CPU Load')
axs[2][0].set_ylabel('Percentage Used')
axs[2][0].plot(df['time_minutes'], df['cpu_load'], alpha=0.3, color="#B2AFAF", linewidth=0.4)
axs[2][0].plot(df['time_minutes'], df['cpu_load_smooth'], alpha=0.3, color="#6E6E6E", linewidth=0.4)
axs[2][0].plot(downsampled_dfs['cpu_load']['time_minutes'], downsampled_dfs['cpu_load']['cpu_load_smooth'], color="#2563EB", linewidth=2.5)

# CPU Temp 
axs[2][1].set_title('CPU Temperature')
axs[2][1].set_ylabel('Temperature (°C)')
axs[3][0].set_ylim(0, 110)
axs[2][1].plot(df['time_minutes'], df['cpu_temp'], alpha=0.3, color="#B2AFAF", linewidth=0.4)
axs[2][1].plot(df['time_minutes'], df['cpu_temp_smooth'], alpha=0.3, color="#6E6E6E", linewidth=0.4)
axs[2][1].plot(downsampled_dfs['cpu_temp']['time_minutes'], downsampled_dfs['cpu_temp']['cpu_temp_smooth'], color="#2563EB", linewidth=2.5)

# GPU Load
axs[3][0].set_title('GPU Load')
axs[3][0].set_ylabel('Percentage Used')
axs[3][0].set_ylim(0, 110)
axs[3][0].plot(df['time_minutes'], df['gpu_load'], alpha=0.3, color="#B2AFAF", linewidth=0.4, label="raw")
axs[3][0].plot(df['time_minutes'], df['gpu_load_smooth'], alpha=0.3, color="#6E6E6E", linewidth=0.4, label="smooth")
axs[3][0].plot(downsampled_dfs['gpu_load']['time_minutes'], downsampled_dfs['gpu_load']['gpu_load_smooth'], color="#2563EB", linewidth=2.5, label="smooth + downsampled")

handles, labels = [], []
for ax in fig.get_axes():
    h, l = ax.get_legend_handles_labels()
    handles.extend(h)
    labels.extend(l)

fig.legend(handles, labels, loc='lower center', ncol=len(labels), bbox_to_anchor=(0.5, -0.05), fancybox=True)

trimmed_name = file[file.find("/")+1:file.find('_')]
plt.savefig(f"{trimmed_name}.png")